# Crisis Connect — Phase 5: Streamlit App + HuggingFace Deployment
**CSC-233 Artificial Intelligence Lab | Spring 2026**
**Component owners: Maleeha Fatima + Abdulraheem Kashif**

### What this notebook does:
1. Installs all required packages
2. Downloads model files from Drive to Colab
3. Tests the Streamlit app locally in Colab
4. Deploys everything to HuggingFace Spaces
5. Gives you a public link to share

**Run order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8**

## Cell 1 — Install packages

In [2]:
!pip install streamlit torch torchvision scikit-learn Pillow huggingface_hub pyngrok nltk -q
!apt-get install -y ffmpeg -q

import nltk
nltk.download('stopwords', quiet=True)

print('All packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 114.5 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
All packages installed.


## Cell 2 — Mount Drive and copy model files to Colab

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Create app folder
APP_DIR = '/content/crisis_connect_app'
os.makedirs(APP_DIR, exist_ok=True)

# Copy model files from Drive to Colab
files_to_copy = {
    '/content/drive/MyDrive/crisis connect_model/resnet50_crisis_connect.pth': 'resnet50_crisis_connect.pth',
    '/content/drive/MyDrive/crisis connect_phase3b/mlp_classifier.pkl':        'mlp_classifier.pkl',
    '/content/drive/MyDrive/crisis connect_phase3a/tfidf_vectorizer.pkl':      'tfidf_vectorizer.pkl',
    '/content/drive/MyDrive/crisis connect_phase3a/label_encoder.pkl':         'label_encoder.pkl',
    '/content/drive/MyDrive/crisis connect_phase4/fusion.py':                  'fusion.py',
}

print('Copying model files to Colab...')
for src, fname in files_to_copy.items():
    dest = os.path.join(APP_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dest)
        size = os.path.getsize(dest) / 1024 / 1024
        print(f'  Copied: {fname}  ({size:.1f} MB)')
    else:
        print(f'  NOT FOUND: {src}')

print()
print('All files ready in:', APP_DIR)

Mounted at /content/drive
Copying model files to Colab...
  Copied: resnet50_crisis_connect.pth  (90.0 MB)
  Copied: mlp_classifier.pkl  (12.6 MB)
  Copied: tfidf_vectorizer.pkl  (0.0 MB)
  Copied: label_encoder.pkl  (0.0 MB)
  Copied: fusion.py  (0.0 MB)

All files ready in: /content/crisis_connect_app


## Cell 3 — Write the Streamlit app

In [4]:
app_code = '''
import streamlit as st
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import joblib
import numpy as np
import re
import os
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)

# ── Page config ─────────────────────────────────────────────
st.set_page_config(
    page_title="Crisis Connect",
    page_icon="🚨",
    layout="centered"
)

# ── Constants ───────────────────────────────────────────────
CLASSES    = ["earthquake", "flood", "fire", "traffic_incident"]
CNN_WEIGHT = 0.70
MLP_WEIGHT = 0.30
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_EMOJI = {
    "earthquake":       "🏚️",
    "flood":            "🌊",
    "fire":             "🔥",
    "traffic_incident": "🚗"
}

SEVERITY_COLOR = {
    "High":   "#FF4444",
    "Medium": "#FFA500",
    "Low":    "#44BB44"
}

# ── Load models (cached so they load only once) ─────────────
@st.cache_resource
def load_all_models():
    base = os.path.dirname(os.path.abspath(__file__))

    # CNN
    cnn = models.resnet50(pretrained=False)
    cnn.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(cnn.fc.in_features, 4))
    ckpt = torch.load(os.path.join(base, "resnet50_crisis_connect.pth"), map_location=device)
    cnn.load_state_dict(ckpt["model_state_dict"])
    cnn = cnn.to(device)
    cnn.eval()
    cnn_idx_to_class = {v: k for k, v in ckpt["class_to_idx"].items()}

    # MLP + tools
    mlp = joblib.load(os.path.join(base, "mlp_classifier.pkl"))
    vec = joblib.load(os.path.join(base, "tfidf_vectorizer.pkl"))
    le  = joblib.load(os.path.join(base, "label_encoder.pkl"))

    return cnn, cnn_idx_to_class, mlp, vec, le

# ── Preprocessing ────────────────────────────────────────────
img_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def clean_text(text):
    sw = set(stopwords.words("english")) - {"no","not","very","too","above","below","near","under","over"}
    text = re.sub(r"[^a-z\\s]", " ", str(text).lower())
    return " ".join([t for t in text.split() if t not in sw and len(t) > 2])

def get_severity(confidence, predicted_class):
    HIGH_RISK = ["earthquake", "flood"]
    if confidence >= 0.75:
        return "High" if predicted_class in HIGH_RISK else "Medium"
    elif confidence >= 0.50:
        return "Medium"
    return "Low"

def predict(image, text, cnn, cnn_idx_to_class, mlp, vec, le):
    # CNN
    tensor = img_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        cnn_raw = torch.softmax(cnn(tensor), dim=1)[0].cpu().numpy()
    cnn_probs = np.zeros(len(CLASSES))
    for idx, cls in cnn_idx_to_class.items():
        if cls in CLASSES:
            cnn_probs[CLASSES.index(cls)] = cnn_raw[idx]
    # MLP
    mlp_raw = mlp.predict_proba(vec.transform([clean_text(text)]))[0]
    mlp_probs = np.zeros(len(CLASSES))
    for i, cls in enumerate(le.classes_):
        if cls in CLASSES:
            mlp_probs[CLASSES.index(cls)] = mlp_raw[i]
    # Fuse
    fused      = CNN_WEIGHT * cnn_probs + MLP_WEIGHT * mlp_probs
    pred_idx   = int(np.argmax(fused))
    pred_class = CLASSES[pred_idx]
    confidence = float(fused[pred_idx])
    return pred_class, confidence, get_severity(confidence, pred_class), cnn_probs, mlp_probs, fused

# ── UI ───────────────────────────────────────────────────────
st.markdown("""
    <h1 style=\'text-align:center; color:#CC0000;\'>🚨 Crisis Connect</h1>
    <p style=\'text-align:center; font-size:16px; color:#666;\'>AI-Powered Disaster Incident Classifier</p>
    <p style=\'text-align:center; font-size:13px; color:#888;\'>Upload a disaster image and describe what you see. The AI will classify the incident and estimate severity.</p>
    <hr/>
""", unsafe_allow_html=True)

# Load models
with st.spinner("Loading AI models... please wait"):
    cnn, cnn_idx_to_class, mlp, vec, le = load_all_models()
st.success("Models loaded. Ready to classify.")

# Input section
col1, col2 = st.columns(2)

with col1:
    st.subheader("📷 Upload Disaster Image")
    uploaded = st.file_uploader(
        "Upload a photo of the incident",
        type=["jpg", "jpeg", "png"]
    )
    if uploaded:
        image = Image.open(uploaded).convert("RGB")
        st.image(image, caption="Uploaded image", use_column_width=True)

with col2:
    st.subheader("📝 Describe the Incident")
    text_input = st.text_area(
        "Describe what you see (max 200 characters)",
        placeholder="e.g. Water rising fast on main road, cars stranded near bridge",
        max_chars=200,
        height=120
    )
    st.caption(f"{len(text_input)}/200 characters")

st.markdown("<br/>", unsafe_allow_html=True)

# Classify button
if st.button("🔍 Classify Incident", use_container_width=True, type="primary"):
    if uploaded is None:
        st.error("Please upload an image first.")
    elif len(text_input.strip()) < 5:
        st.error("Please enter a description of at least 5 characters.")
    else:
        with st.spinner("Analysing..."):
            pred_class, confidence, severity, cnn_probs, mlp_probs, fused = predict(
                image, text_input, cnn, cnn_idx_to_class, mlp, vec, le
            )

        st.markdown("<hr/>", unsafe_allow_html=True)
        st.subheader("🎯 Classification Result")

        # Main result
        sev_color = SEVERITY_COLOR[severity]
        emoji     = CLASS_EMOJI.get(pred_class, "⚠️")
        st.markdown(f"""
            <div style=\'background:#f8f8f8; border-radius:12px; padding:20px; text-align:center; border: 2px solid {sev_color};\' >
                <h2>{emoji} {pred_class.replace("_", " ").title()}</h2>
                <p style=\'font-size:18px;\'>Confidence: <b>{confidence*100:.1f}%</b></p>
                <p style=\'font-size:20px; color:{sev_color}; font-weight:bold;\'>Severity: {severity}</p>
            </div>
        """, unsafe_allow_html=True)

        st.markdown("<br/>", unsafe_allow_html=True)

        # Probability bars
        st.subheader("📊 Class Probabilities")
        tabs = st.tabs(["Fused (Final)", "CNN Only", "Text Only"])

        for tab, probs, label in zip(
            tabs,
            [fused, cnn_probs, mlp_probs],
            ["Fused (CNN 70% + Text 30%)", "CNN Image Model", "Text MLP Model"]
        ):
            with tab:
                st.caption(label)
                for i, cls in enumerate(CLASSES):
                    emoji_cls = CLASS_EMOJI.get(cls, "")
                    st.progress(
                        float(probs[i]),
                        text=f"{emoji_cls} {cls.replace(\'_\', \' \').title()}: {probs[i]*100:.1f}%"
                    )

        st.markdown("<br/>", unsafe_allow_html=True)
        st.info("Powered by ResNet-50 (CNN) + TF-IDF MLP (Text) fusion — Crisis Connect AI Lab Spring 2026")

# Footer
st.markdown("<hr/>", unsafe_allow_html=True)
st.markdown("""
    <p style=\'text-align:center; font-size:12px; color:#aaa;\'>\n    Crisis Connect | CSC-233 AI Lab | Spring 2026 | Beaconhouse National University\n    </p>
""", unsafe_allow_html=True)
'''

app_path = os.path.join(APP_DIR, 'app.py')
with open(app_path, 'w') as f:
    f.write(app_code)

print(f'Streamlit app written to: {app_path}')
print(f'App size: {os.path.getsize(app_path)/1024:.1f} KB')

Streamlit app written to: /content/crisis_connect_app/app.py
App size: 7.4 KB


## Cell 4 — Write requirements.txt

In [5]:
requirements = """streamlit
torch
torchvision
scikit-learn
Pillow
nltk
numpy
joblib
"""

req_path = os.path.join(APP_DIR, 'requirements.txt')
with open(req_path, 'w') as f:
    f.write(requirements)

print('requirements.txt written.')
print()
print(open(req_path).read())

requirements.txt written.

streamlit
torch
torchvision
scikit-learn
Pillow
nltk
numpy
joblib



In [6]:
from google.colab import files
import os

APP_DIR = '/content/crisis_connect_app'

for fname in ['app.py', 'requirements.txt', 'fusion.py']:
    fpath = os.path.join(APP_DIR, fname)
    if os.path.exists(fpath):
        print(f'Downloading {fname}...')
        files.download(fpath)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 5 — Test app locally in Colab using ngrok

In [7]:
# Get your ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
# Sign up free, copy your token, paste it below
NGROK_TOKEN = '3EE0AyZl7zG8WTxZGUhzb4CUDnv_5urchUiCtCZbmJFsPiZu5'

from pyngrok import ngrok
import subprocess, time

ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing streamlit
!pkill -f streamlit 2>/dev/null
time.sleep(1)

# Start streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', f'{APP_DIR}/app.py',
     '--server.port', '8501',
     '--server.headless', 'true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)

# Open tunnel
tunnel = ngrok.connect(8501)
print('='*50)
print('Streamlit app is running!')
print(f'Open this link: {tunnel.public_url}')
print('='*50)
print()
print('Test the app in your browser.')
print('When done testing, run Cell 6 to deploy to HuggingFace.')

^C
Streamlit app is running!
Open this link: https://wobbling-pretender-cash.ngrok-free.dev

Test the app in your browser.
When done testing, run Cell 6 to deploy to HuggingFace.


In [12]:
import subprocess

print('GitHub Repository Details:')
print()
print('Repository Name : crisis-connect')
print('Repository URL  : https://github.com/f2024-0693-blip/crisis-connect')
print('Branch          : main')
print()
print('Files in repository:')
print('  app.py              Main Streamlit application')
print('  requirements.txt    Python dependencies')
print()
print('Deployment Connection:')
print('  Platform    : Streamlit Cloud')
print('  Connected to: GitHub repo crisis-connect')
print('  Auto-deploy : Yes — any push to main branch redeploys the app')
print('  Live URL    : https://crisis-connect-vjmeydb54qdgg7yxm3xfes.streamlit.app/')

GitHub Repository Details:

Repository Name : crisis-connect
Repository URL  : https://github.com/f2024-0693-blip/crisis-connect
Branch          : main

Files in repository:
  app.py              Main Streamlit application
  requirements.txt    Python dependencies

Deployment Connection:
  Platform    : Streamlit Cloud
  Connected to: GitHub repo crisis-connect
  Auto-deploy : Yes — any push to main branch redeploys the app
  Live URL    : https://crisis-connect-vjmeydb54qdgg7yxm3xfes.streamlit.app/


In [11]:
print('=' * 55)
print('  Phase 5 Complete — Crisis Connect App Deployed')
print('=' * 55)
print()
print('Deployment Platform : Streamlit Cloud')
print('https://crisis-connect-vjmeydb54qdgg7yxm3xfes.streamlit.app/')
print()
print('What was built:')
print('  app.py              Streamlit frontend')
print('  requirements.txt    Dependencies')
print()
print('How the app works:')
print('  1. User uploads a disaster photo')
print('  2. User types a short description')
print('  3. CNN classifies the image (70% weight)')
print('  4. MLP classifies the text  (30% weight)')
print('  5. Fused result: class + confidence + severity')
print()
print('All phases complete:')
print('  Phase 1  Dataset preparation              DONE')
print('  Phase 2  ResNet-50 CNN       91.03%       DONE')
print('  Phase 3a TF-IDF preprocessing             DONE')
print('  Phase 3b MLP text classifier  73.33%      DONE')
print('  Phase 4  Fusion layer         94.17%      DONE')
print('  Phase 5  Streamlit Cloud app              DONE')

  Phase 5 Complete — Crisis Connect App Deployed

Deployment Platform : Streamlit Cloud
https://crisis-connect-vjmeydb54qdgg7yxm3xfes.streamlit.app/

What was built:
  app.py              Streamlit frontend
  requirements.txt    Dependencies

How the app works:
  1. User uploads a disaster photo
  2. User types a short description
  3. CNN classifies the image (70% weight)
  4. MLP classifies the text  (30% weight)
  5. Fused result: class + confidence + severity

All phases complete:
  Phase 1  Dataset preparation              DONE
  Phase 2  ResNet-50 CNN       91.03%       DONE
  Phase 3a TF-IDF preprocessing             DONE
  Phase 3b MLP text classifier  73.33%      DONE
  Phase 4  Fusion layer         94.17%      DONE
  Phase 5  Streamlit Cloud app              DONE


## Cell 6 — Summary

In [ ]:
print('=' * 55)
print('  Phase 5 Complete — Crisis Connect App Deployed')
print('=' * 55)
print()
print('What was built:')
print('  app.py              Streamlit frontend')
print('  requirements.txt    Dependencies for HuggingFace')
print()
print('How the app works:')
print('  1. User uploads a disaster photo')
print('  2. User types a short description')
print('  3. CNN classifies the image (70% weight)')
print('  4. MLP classifies the text (30% weight)')
print('  5. Fused result shows: class + confidence + severity')
print('  6. Probability bars show CNN vs Text vs Fused scores')
print()
print('All phases complete:')
print('  Phase 1  Dataset preparation         DONE')
print('  Phase 2  ResNet-50 CNN  91.03%        DONE')
print('  Phase 3a TF-IDF preprocessing         DONE')
print('  Phase 3b MLP text classifier  73.33%  DONE')
print('  Phase 4  Fusion layer   94.17%        DONE')
print('  Phase 5  Streamlit + HuggingFace      DONE')